# 🎓 Модель распознавания негатива в диалогах

## Архитектура модели:
1. **BERT Embedder Layer** — кодирует каждую реплику диалога в sentence embedding
2. **Self-Attention** — контекстуализирует реплики с учётом всего диалога
3. **Attention Pooling** — агрегирует контекстуализированные эмбеддинги в один вектор
4. **Dot Product** — сравнивает вектор диалога с обучаемым вектором класса «негатив»
5. **Threshold** — принимает бинарное решение

---

## 1. Установка зависимостей

In [1]:
!pip install -q torch transformers scikit-learn numpy tqdm matplotlib seaborn


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python3.10 -m pip install --upgrade pip


## 2. Импорты

In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass

# Проверяем доступность GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Устройство: cpu


## 3. Конфигурация

In [3]:
@dataclass
class ModelConfig:
    """Параметры модели"""
    bert_model_name: str = "cointegrated/rubert-tiny2"  # Лёгкая русскоязычная BERT
    sentence_embedding_dim: int = 312  # Размерность выхода rubert-tiny2
    num_attention_heads: int = 4  # Число голов в self-attention
    attention_dropout: float = 0.1
    max_dialog_length: int = 10  # Максимальное число реплик в диалоге
    freeze_bert: bool = False  # Замораживать ли веса BERT


@dataclass
class TrainingConfig:
    """Параметры обучения"""
    batch_size: int = 8
    learning_rate: float = 2e-5
    num_epochs: int = 15
    weight_decay: float = 0.01
    threshold: float = 0.5
    seed: int = 42
    val_split: float = 0.2


model_config = ModelConfig()
train_config = TrainingConfig()

print("Конфигурация модели:")
print(f"  BERT: {model_config.bert_model_name}")
print(f"  Embedding dim: {model_config.sentence_embedding_dim}")
print(f"  Attention heads: {model_config.num_attention_heads}")
print(f"  Max dialog length: {model_config.max_dialog_length}")
print(f"\nКонфигурация обучения:")
print(f"  Batch size: {train_config.batch_size}")
print(f"  Learning rate: {train_config.learning_rate}")
print(f"  Epochs: {train_config.num_epochs}")
print(f"  Threshold: {train_config.threshold}")

Конфигурация модели:
  BERT: cointegrated/rubert-tiny2
  Embedding dim: 312
  Attention heads: 4
  Max dialog length: 10

Конфигурация обучения:
  Batch size: 8
  Learning rate: 2e-05
  Epochs: 15
  Threshold: 0.5


## 4. Данные для обучения

Синтетические данные — диалоги из чатов поддержки с метками: 
- `1` — негативный диалог
- `0` — нейтральный/позитивный диалог

In [4]:
SYNTHETIC_DIALOGS = [
    # ==================== НЕГАТИВНЫЕ ДИАЛОГИ ====================
    {
        "sentences": [
            "Здравствуйте, где мой заказ?",
            "Мы уточняем информацию, подождите",
            "Я жду уже неделю! Это ужасный сервис!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Добрый день, у меня проблема с товаром",
            "Опишите проблему подробнее",
            "Товар пришёл сломанный, коробка мятая",
            "Мы можем предложить замену",
            "Нет, верните деньги. Я больше не буду у вас заказывать",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Почему мне до сих пор не перезвонили?",
            "Извините за задержку",
            "Вы вообще работаете там? Безобразие!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Я оплатил заказ три дня назад",
            "Статус вашего заказа: обрабатывается",
            "Три дня обрабатывается?! Вы издеваетесь?!",
            "Приносим извинения за неудобства",
            "Мне не нужны извинения, мне нужен мой товар!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Здравствуйте, хочу вернуть товар",
            "К сожалению, возврат невозможен после 14 дней",
            "Это незаконно! Я буду жаловаться!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Ваш сайт опять не работает",
            "Попробуйте очистить кэш браузера",
            "Я уже пробовал, ничего не помогает. Сделайте нормальный сайт!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Курьер не приехал вовремя",
            "Приносим извинения, курьер задерживается",
            "Задерживается на 3 часа?! Это неприемлемо!",
            "Мы сделаем скидку на следующий заказ",
            "Мне не нужна скидка, мне нужна нормальная доставка!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Опять списали деньги дважды",
            "Мы проверим транзакцию",
            "Каждый раз одно и то же! Когда это закончится?!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Ваш менеджер был очень грубый",
            "Мы разберёмся в ситуации",
            "Разбирайтесь быстрее, я устал от вашего отношения!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Товар не соответствует описанию!",
            "Можете прислать фото?",
            "Вот фото. Это совсем не то, что на сайте! Обман!",
            "Мы оформим возврат",
            "Я теряю время из-за вашей некомпетентности",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Привет, не могу войти в аккаунт",
            "Мы заблокировали аккаунт из-за подозрительной активности",
            "Какая активность? Это мой аккаунт! Разблокируйте немедленно!",
            "Нужно пройти верификацию",
            "Это бред! Я просто хочу зайти на свою страницу!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Подписка списалась автоматически",
            "Вы можете отменить в настройках",
            "Я не подписывался! Верните деньги! Это мошенничество!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Мне привезли не тот цвет",
            "Мы можем организовать обмен",
            "Обмен?! Я ждал 2 недели! Это кошмар!",
            "Понимаем ваше разочарование",
            "Да что вы понимаете? Отвратительно работаете!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "У вас ошибка в приложении, не могу оформить заказ",
            "Попробуйте обновить приложение",
            "Обновил, не помогло! Сколько можно это терпеть?!",
        ],
        "label": 1,
    },
    {
        "sentences": [
            "Вы обещали доставку сегодня",
            "К сожалению, произошла задержка на складе",
            "Меня не интересуют ваши проблемы! Где мой заказ?!",
            "Завтра курьер доставит",
            "Каждый раз одни обещания! Я напишу жалобу!",
        ],
        "label": 1,
    },
    # ==================== НЕЙТРАЛЬНЫЕ / ПОЗИТИВНЫЕ ДИАЛОГИ ====================
    {
        "sentences": [
            "Здравствуйте, подскажите статус заказа",
            "Ваш заказ отправлен, трек-номер: ABC123",
            "Спасибо большое!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Добрый день! Хочу узнать о доставке",
            "Доставка занимает 2-3 рабочих дня",
            "Отлично, спасибо за информацию!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Подскажите, есть ли этот товар в наличии?",
            "Да, товар в наличии. Оформить заказ?",
            "Да, пожалуйста. Спасибо!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Здравствуйте, хотел бы изменить адрес доставки",
            "Конечно, назовите новый адрес",
            "ул. Ленина, 15, кв. 42",
            "Адрес изменён. Что-то ещё могу помочь?",
            "Нет, это всё. Спасибо!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Привет! Какие у вас способы оплаты?",
            "Картой, наличными при получении, через СБП",
            "Понял, спасибо!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Добрый вечер, заказ получил",
            "Рады слышать! Всё в порядке с товаром?",
            "Да, всё отлично! Качество супер!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Здравствуйте, можно узнать время работы?",
            "Работаем с 9 до 21 ежедневно",
            "Спасибо, приду завтра",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Хочу оформить подарочный сертификат",
            "На какую сумму?",
            "На 3000 рублей",
            "Готово! Отправить на email?",
            "Да, пожалуйста. Спасибо!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Подскажите размерную сетку",
            "Для этой модели рекомендуем брать на размер больше",
            "Хорошо, тогда возьму L. Спасибо за совет!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Можно ли оформить рассрочку?",
            "Да, доступна рассрочка на 6 месяцев без процентов",
            "Отлично, оформите, пожалуйста",
            "Готово! Первый платёж через месяц",
            "Супер, спасибо за помощь!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Привет, получил заказ сегодня",
            "Всё ли в порядке?",
            "Да, доставили быстро, упаковка целая. Спасибо!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Здравствуйте, хочу записаться на консультацию",
            "Свободные слоты: среда 14:00, пятница 10:00",
            "Давайте в среду в 14:00",
            "Записал вас. Ждём!",
            "Спасибо, до встречи!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Добрый день, есть ли скидки для постоянных клиентов?",
            "Да, у вас накопительная скидка 10%",
            "Здорово, не знал! Спасибо!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Хочу уточнить характеристики товара",
            "Конечно, что именно интересует?",
            "Размер экрана и вес",
            "Экран 6.5 дюймов, вес 185 грамм",
            "Идеально, заказываю!",
        ],
        "label": 0,
    },
    {
        "sentences": [
            "Подскажите, как активировать промокод?",
            "Введите его в поле при оформлении заказа",
            "Получилось, скидка применилась. Спасибо!",
        ],
        "label": 0,
    },
]

# Статистика
neg_count = sum(1 for d in SYNTHETIC_DIALOGS if d["label"] == 1)
pos_count = sum(1 for d in SYNTHETIC_DIALOGS if d["label"] == 0)
print(f"Всего диалогов: {len(SYNTHETIC_DIALOGS)}")
print(f"  Негативных: {neg_count}")
print(f"  Нейтральных/позитивных: {pos_count}")

Всего диалогов: 30
  Негативных: 15
  Нейтральных/позитивных: 15


## 5. Датасет и DataLoader

In [5]:
class DialogDataset(Dataset):
    """
    Датасет диалогов для обучения модели.
    Каждый элемент — диалог из нескольких реплик и метка (0/1).
    """

    def __init__(self, dialogs: list, tokenizer_name: str, max_token_length: int = 128):
        self.dialogs = dialogs
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_token_length = max_token_length

    def __len__(self):
        return len(self.dialogs)

    def __getitem__(self, idx):
        dialog = self.dialogs[idx]
        sentences = dialog["sentences"]
        label = dialog["label"]

        # Токенизируем каждую реплику
        encoded_sentences = []
        for sentence in sentences:
            encoded = self.tokenizer(
                sentence,
                padding="max_length",
                truncation=True,
                max_length=self.max_token_length,
                return_tensors="pt",
            )
            encoded_sentences.append(
                {
                    "input_ids": encoded["input_ids"].squeeze(0),
                    "attention_mask": encoded["attention_mask"].squeeze(0),
                }
            )

        return {
            "encoded_sentences": encoded_sentences,
            "label": torch.tensor(label, dtype=torch.float),
            "num_sentences": len(sentences),
            "raw_sentences": sentences,
        }


def collate_fn(batch):
    """
    Собирает батч из диалогов разной длины.
    """
    all_input_ids = []
    all_attention_masks = []
    labels = []
    num_sentences_per_dialog = []
    raw_sentences_list = []

    max_sentences = max(item["num_sentences"] for item in batch)

    for item in batch:
        for encoded in item["encoded_sentences"]:
            all_input_ids.append(encoded["input_ids"])
            all_attention_masks.append(encoded["attention_mask"])

        labels.append(item["label"])
        num_sentences_per_dialog.append(item["num_sentences"])
        raw_sentences_list.append(item["raw_sentences"])

    all_input_ids = torch.stack(all_input_ids)
    all_attention_masks = torch.stack(all_attention_masks)
    labels = torch.stack(labels)

    # Маска диалога
    batch_size = len(batch)
    dialog_mask = torch.zeros(batch_size, max_sentences)
    for i, num_sent in enumerate(num_sentences_per_dialog):
        dialog_mask[i, :num_sent] = 1.0

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": labels,
        "num_sentences_per_dialog": num_sentences_per_dialog,
        "dialog_mask": dialog_mask,
        "raw_sentences": raw_sentences_list,
    }


print("✓ Классы датасета определены")

✓ Классы датасета определены


## 6. Архитектура модели

Пайплайн:
```
Реплики диалога
    → BERT Embedder (sentence embeddings)
    → Positional Encoding
    → Self-Attention между репликами
    → Contextualized Sentence Embeddings
    → Attention Pooling
    → Dot Product с вектором класса
    → Threshold → Решение
```

In [6]:
class BERTSentenceEncoder(nn.Module):
    """
    Кодирует каждую реплику диалога в sentence embedding.
    Используется mean pooling по токенам.
    """

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.bert = AutoModel.from_pretrained(config.bert_model_name)

        if config.freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

    def mean_pooling(self, model_output, attention_mask):
        """Mean pooling с учётом маски"""
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = (
            attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        )
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sentence_embeddings = self.mean_pooling(outputs, attention_mask)
        return sentence_embeddings


class DialogSelfAttention(nn.Module):
    """
    Multi-Head Self-Attention между репликами диалога.
    Позволяет каждой реплике учитывать контекст остальных.
    """

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.embed_dim = config.sentence_embedding_dim
        self.num_heads = config.num_attention_heads

        # Позиционное кодирование (порядок реплик в диалоге)
        self.position_embedding = nn.Embedding(
            config.max_dialog_length, self.embed_dim
        )

        # Multi-Head Self-Attention
        self.self_attention = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=self.num_heads,
            dropout=config.attention_dropout,
            batch_first=True,
        )

        # Layer Normalization
        self.layer_norm1 = nn.LayerNorm(self.embed_dim)
        self.layer_norm2 = nn.LayerNorm(self.embed_dim)

        # Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, self.embed_dim * 4),
            nn.GELU(),
            nn.Dropout(config.attention_dropout),
            nn.Linear(self.embed_dim * 4, self.embed_dim),
            nn.Dropout(config.attention_dropout),
        )

    def forward(self, sentence_embeddings, dialog_mask):
        batch_size, num_sentences, _ = sentence_embeddings.shape

        # Позиционные эмбеддинги
        positions = torch.arange(num_sentences, device=sentence_embeddings.device)
        positions = positions.unsqueeze(0).expand(batch_size, -1)
        pos_embeddings = self.position_embedding(positions)
        x = sentence_embeddings + pos_embeddings

        # Key padding mask (True = игнорировать)
        key_padding_mask = ~dialog_mask.bool()

        # Self-Attention + Residual + LayerNorm
        attn_output, attn_weights = self.self_attention(
            query=x, key=x, value=x, key_padding_mask=key_padding_mask
        )
        x = self.layer_norm1(x + attn_output)

        # FFN + Residual + LayerNorm
        ffn_output = self.ffn(x)
        x = self.layer_norm2(x + ffn_output)

        return x, attn_weights


class AttentionPooling(nn.Module):
    """
    Attention Pooling — агрегация эмбеддингов реплик
    в один вектор диалога с обучаемыми весами.
    """

    def __init__(self, embed_dim):
        super().__init__()
        self.attention_vector = nn.Linear(embed_dim, 1)

    def forward(self, contextualized_embeddings, dialog_mask):
        # Скоры внимания
        scores = self.attention_vector(contextualized_embeddings).squeeze(-1)

        # Маскируем паддинг
        scores = scores.masked_fill(~dialog_mask.bool(), float("-inf"))

        # Softmax
        weights = torch.softmax(scores, dim=-1)

        # Взвешенная сумма
        dialog_embedding = torch.bmm(
            weights.unsqueeze(1), contextualized_embeddings
        ).squeeze(1)

        return dialog_embedding, weights


class NegativeDetectionModel(nn.Module):
    """
    Полная модель распознавания негатива в диалогах.

    Архитектура:
        1. BERT Sentence Encoder
        2. Dialog Self-Attention
        3. Attention Pooling
        4. Dot Product с вектором класса
        5. Threshold
    """

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        # 1. BERT Sentence Encoder
        self.sentence_encoder = BERTSentenceEncoder(config)

        # 2. Dialog Self-Attention
        self.dialog_attention = DialogSelfAttention(config)

        # 3. Attention Pooling
        self.attention_pooling = AttentionPooling(config.sentence_embedding_dim)

        # 4. Обучаемый вектор класса "негатив"
        self.negative_class_vector = nn.Parameter(
            torch.randn(config.sentence_embedding_dim)
        )

        # Нормализация
        self.output_norm = nn.LayerNorm(config.sentence_embedding_dim)

    def forward(self, input_ids, attention_mask, num_sentences_per_dialog, dialog_mask):
        batch_size = len(num_sentences_per_dialog)
        max_sentences = dialog_mask.shape[1]

        # 1. Sentence embeddings через BERT
        all_sentence_embeddings = self.sentence_encoder(input_ids, attention_mask)

        # 2. Раскладываем по диалогам
        dialog_embeddings = torch.zeros(
            batch_size,
            max_sentences,
            self.config.sentence_embedding_dim,
            device=input_ids.device,
        )

        idx = 0
        for i, num_sent in enumerate(num_sentences_per_dialog):
            dialog_embeddings[i, :num_sent] = all_sentence_embeddings[idx : idx + num_sent]
            idx += num_sent

        # 3. Self-Attention → contextualized embeddings
        contextualized, attn_weights = self.dialog_attention(
            dialog_embeddings, dialog_mask
        )

        # 4. Attention Pooling → один вектор на диалог
        dialog_vector, pooling_weights = self.attention_pooling(
            contextualized, dialog_mask
        )

        # 5. Нормализация
        dialog_vector = self.output_norm(dialog_vector)

        # 6. Dot Product (cosine similarity)
        neg_vector = torch.nn.functional.normalize(
            self.negative_class_vector.unsqueeze(0), dim=-1
        )
        dialog_vector_norm = torch.nn.functional.normalize(dialog_vector, dim=-1)

        # Cosine similarity → [0, 1]
        scores = torch.sum(dialog_vector_norm * neg_vector, dim=-1)
        scores = (scores + 1) / 2

        return scores, attn_weights, pooling_weights

    def predict(self, scores, threshold=0.5):
        """Применяет threshold"""
        return (scores > threshold).long()


print("✓ Архитектура модели определена")

✓ Архитектура модели определена


## 7. Подготовка к обучению

In [7]:
def set_seed(seed):
    """Фиксируем seed для воспроизводимости"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def split_data(dialogs, val_split, seed):
    """Разбиение данных на train и val"""
    random.seed(seed)
    indices = list(range(len(dialogs)))
    random.shuffle(indices)

    val_size = int(len(dialogs) * val_split)
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    train_dialogs = [dialogs[i] for i in train_indices]
    val_dialogs = [dialogs[i] for i in val_indices]

    return train_dialogs, val_dialogs


# Фиксируем seed
set_seed(train_config.seed)

# Разделяем данные
train_dialogs, val_dialogs = split_data(
    SYNTHETIC_DIALOGS, train_config.val_split, train_config.seed
)

print(f"Train: {len(train_dialogs)} диалогов")
print(f"Val: {len(val_dialogs)} диалогов")
print(f"  Train негатив: {sum(1 for d in train_dialogs if d['label'] == 1)}")
print(f"  Train нейтрально: {sum(1 for d in train_dialogs if d['label'] == 0)}")
print(f"  Val негатив: {sum(1 for d in val_dialogs if d['label'] == 1)}")
print(f"  Val нейтрально: {sum(1 for d in val_dialogs if d['label'] == 0)}")

Train: 24 диалогов
Val: 6 диалогов
  Train негатив: 12
  Train нейтрально: 12
  Val негатив: 3
  Val нейтрально: 3


In [8]:
# Создаём датасеты и DataLoader
train_dataset = DialogDataset(train_dialogs, model_config.bert_model_name)
val_dataset = DialogDataset(val_dialogs, model_config.bert_model_name)

train_loader = DataLoader(
    train_dataset,
    batch_size=train_config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=train_config.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Train batches: 3
Val batches: 1


In [9]:
# Создаём модель
model = NegativeDetectionModel(model_config).to(device)

# Считаем параметры
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*50}")
print(f"МОДЕЛЬ СОЗДАНА")
print(f"{'='*50}")
print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")
print(f"\nСтруктура модели:")
print(model)

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



МОДЕЛЬ СОЗДАНА
Всего параметров: 30,370,321
Обучаемых параметров: 30,370,321

Структура модели:
NegativeDetectionModel(
  (sentence_encoder): BERTSentenceEncoder(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(83828, 312, padding_idx=0)
        (position_embeddings): Embedding(2048, 312)
        (token_type_embeddings): Embedding(2, 312)
        (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-2): 3 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=312, out_features=312, bias=True)
                (key): Linear(in_features=312, out_features=312, bias=True)
                (value): Linear(in_features=312, out_features=312, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
 

## 8. Обучение модели

In [10]:
def train_epoch(model, dataloader, optimizer, device):
    """Одна эпоха обучения"""
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    criterion = nn.BCELoss()

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        dialog_mask = batch["dialog_mask"].to(device)
        num_sentences = batch["num_sentences_per_dialog"]

        # Forward
        scores, _, _ = model(input_ids, attention_mask, num_sentences, dialog_mask)

        # Loss
        loss = criterion(scores, labels)
        total_loss += loss.item()

        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Predictions
        preds = model.predict(scores, threshold=train_config.threshold)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)

    return avg_loss, accuracy


def evaluate(model, dataloader, device, threshold=0.5):
    """Валидация модели"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_scores = []

    criterion = nn.BCELoss()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            dialog_mask = batch["dialog_mask"].to(device)
            num_sentences = batch["num_sentences_per_dialog"]

            scores, _, _ = model(input_ids, attention_mask, num_sentences, dialog_mask)

            loss = criterion(scores, labels)
            total_loss += loss.item()

            preds = model.predict(scores, threshold=threshold)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_scores.extend(scores.cpu().numpy())

    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0
    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
    }

    return metrics, all_preds, all_labels, all_scores


print("✓ Функции обучения и валидации определены")

✓ Функции обучения и валидации определены


In [ ]:
# Оптимизатор
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_config.learning_rate,
    weight_decay=train_config.weight_decay,
)

# История обучения
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
    "val_precision": [],
    "val_recall": [],
}

best_f1 = 0

print("="*60)
print("НАЧАЛО ОБУЧЕНИЯ")
print("="*60)
print(f"Эпох: {train_config.num_epochs}")
print(f"Batch size: {train_config.batch_size}")
print(f"Learning rate: {train_config.learning_rate}")
print(f"Threshold: {train_config.threshold}")
print("="*60)

for epoch in range(train_config.num_epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)

    # Validate
    val_metrics, val_preds, val_labels, val_scores = evaluate(
        model, val_loader, device, train_config.threshold
    )

    # Сохраняем историю
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])

    # Вывод
    print(
        f"Эпоха {epoch+1:2d}/{train_config.num_epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    # Сохраняем лучшую модель
    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        best_model_state = model.state_dict().copy()
        print(f"         ✓ Лучшая модель (F1: {best_f1:.4f})")

print("\n" + "="*60)
print(f"ОБУЧЕНИЕ ЗАВЕРШЕНО. Лучший F1: {best_f1:.4f}")
print("="*60)

НАЧАЛО ОБУЧЕНИЯ
Эпох: 15
Batch size: 8
Learning rate: 2e-05
Threshold: 0.5
Эпоха  1/15 | Train Loss: 0.6898 | Train Acc: 0.5833 | Val Loss: 0.6789 | Val Acc: 0.6667 | Val F1: 0.5000
         ✓ Лучшая модель (F1: 0.5000)
Эпоха  2/15 | Train Loss: 0.6755 | Train Acc: 0.7917 | Val Loss: 0.6704 | Val Acc: 0.8333 | Val F1: 0.8571
         ✓ Лучшая модель (F1: 0.8571)
Эпоха  3/15 | Train Loss: 0.6692 | Train Acc: 0.7500 | Val Loss: 0.6618 | Val Acc: 0.8333 | Val F1: 0.8571
Эпоха  4/15 | Train Loss: 0.6567 | Train Acc: 0.9583 | Val Loss: 0.6526 | Val Acc: 1.0000 | Val F1: 1.0000
         ✓ Лучшая модель (F1: 1.0000)
Эпоха  5/15 | Train Loss: 0.6400 | Train Acc: 1.0000 | Val Loss: 0.6432 | Val Acc: 1.0000 | Val F1: 1.0000
Эпоха  6/15 | Train Loss: 0.6276 | Train Acc: 1.0000 | Val Loss: 0.6327 | Val Acc: 1.0000 | Val F1: 1.0000
Эпоха  7/15 | Train Loss: 0.6171 | Train Acc: 0.9583 | Val Loss: 0.6211 | Val Acc: 1.0000 | Val F1: 1.0000
Эпоха  8/15 | Train Loss: 0.5996 | Train Acc: 1.0000 | Val Los

## 9. Визуализация результатов обучения

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history["train_loss"], label="Train Loss", marker="o", linewidth=2)
axes[0, 0].plot(history["val_loss"], label="Val Loss", marker="s", linewidth=2)
axes[0, 0].set_xlabel("Эпоха")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Loss по эпохам")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history["train_acc"], label="Train Accuracy", marker="o", linewidth=2)
axes[0, 1].plot(history["val_acc"], label="Val Accuracy", marker="s", linewidth=2)
axes[0, 1].set_xlabel("Эпоха")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].set_title("Accuracy по эпохам")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# F1
axes[1, 0].plot(history["val_f1"], label="Val F1", marker="s", linewidth=2, color="green")
axes[1, 0].set_xlabel("Эпоха")
axes[1, 0].set_ylabel("F1 Score")
axes[1, 0].set_title("F1 Score по эпохам (валидация)")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Precision & Recall
axes[1, 1].plot(history["val_precision"], label="Precision", marker="o", linewidth=2)
axes[1, 1].plot(history["val_recall"], label="Recall", marker="s", linewidth=2)
axes[1, 1].set_xlabel("Эпоха")
axes[1, 1].set_ylabel("Score")
axes[1, 1].set_title("Precision и Recall по эпохам (валидация)")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Финальная оценка на валидации

In [ ]:
# Загружаем лучшую модель
if best_f1 > 0:
    model.load_state_dict(best_model_state)
    print("✓ Загружена лучшая модель\n")

# Финальная оценка
val_metrics, val_preds, val_labels, val_scores = evaluate(
    model, val_loader, device, train_config.threshold
)

print("="*50)
print("ФИНАЛЬНЫЕ МЕТРИКИ НА ВАЛИДАЦИИ")
print("="*50)
print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
print(f"  Precision: {val_metrics['precision']:.4f}")
print(f"  Recall:    {val_metrics['recall']:.4f}")
print(f"  F1 Score:  {val_metrics['f1']:.4f}")
print()

# Classification Report
print("Подробный отчёт:")
print(classification_report(
    val_labels, val_preds,
    target_names=["Нейтральный", "Негативный"],
    zero_division=0
))

In [ ]:
# Матрица ошибок
if len(set(val_labels)) > 1:  # Проверяем что есть оба класса
    cm = confusion_matrix(val_labels, val_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Нейтральный", "Негативный"],
        yticklabels=["Нейтральный", "Негативный"],
        annot_kws={"size": 16},
    )
    plt.xlabel("Предсказание", fontsize=12)
    plt.ylabel("Истинная метка", fontsize=12)
    plt.title("Матрица ошибок", fontsize=14)
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Недостаточно классов на валидации для построения матрицы ошибок")

## 11. Демонстрация на тестовых диалогах

In [ ]:
def predict_dialog(model, sentences, tokenizer, device, threshold=0.5):
    """
    Предсказывает негативность диалога.
    """
    model.eval()

    # Токенизируем
    all_input_ids = []
    all_attention_masks = []

    for sentence in sentences:
        encoded = tokenizer(
            sentence,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        all_input_ids.append(encoded["input_ids"].squeeze(0))
        all_attention_masks.append(encoded["attention_mask"].squeeze(0))

    input_ids = torch.stack(all_input_ids).to(device)
    attention_mask = torch.stack(all_attention_masks).to(device)
    dialog_mask = torch.ones(1, len(sentences)).to(device)

    with torch.no_grad():
        scores, attn_weights, pooling_weights = model(
            input_ids, attention_mask, [len(sentences)], dialog_mask
        )

    score = scores.item()
    label = 1 if score > threshold else 0

    return {
        "score": score,
        "label": label,
        "label_text": "❌ НЕГАТИВ" if label == 1 else "✅ НЕЙТРАЛЬНО",
        "pooling_weights": pooling_weights.cpu().numpy()[0],
    }


# Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_config.bert_model_name)

print("✓ Функция предсказания готова")

In [ ]:
# Тестовые диалоги (не из обучающей выборки!)
test_dialogs = [
    {
        "title": "Негативный диалог — жалоба на доставку",
        "sentences": [
            "Здравствуйте, мой заказ не пришёл",
            "Уточняем информацию, ожидайте",
            "Я жду уже 2 недели! Это безобразие! Верните деньги!",
        ],
    },
    {
        "title": "Нейтральный диалог — вопрос о доставке",
        "sentences": [
            "Добрый день, подскажите стоимость доставки",
            "Доставка бесплатна от 3000 рублей",
            "Отлично, спасибо!",
        ],
    },
    {
        "title": "Негативный диалог — сломанный товар",
        "sentences": [
            "Ваш товар сломался через день!",
            "Обратитесь в сервисный центр",
            "Это ваша проблема, а не моя! Я требую замену!",
            "К сожалению, замена не предусмотрена",
            "Ужасный магазин! Больше никогда!",
        ],
    },
    {
        "title": "Нейтральный диалог — ожидание товара",
        "sentences": [
            "Привет! Когда появится товар в наличии?",
            "Ожидаем поставку на следующей неделе",
            "Хорошо, тогда подожду. Спасибо!",
        ],
    },
    {
        "title": "Негативный диалог — проблемы с оплатой",
        "sentences": [
            "С меня списали деньги дважды!",
            "Мы рассмотрим обращение в течение 3 дней",
            "Три дня?! Это мои деньги! Верните немедленно!",
            "Нужно дождаться подтверждения от банка",
            "Я устал ждать! Я обращусь в суд!",
        ],
    },
    {
        "title": "Нейтральный диалог — покупка подарка",
        "sentences": [
            "Здравствуйте, помогите выбрать подарок",
            "Для кого подарок? Мужчина или женщина?",
            "Для жены, на день рождения",
            "Рекомендую набор косметики или украшение",
            "Отличная идея, возьму набор! Спасибо!",
        ],
    },
    {
        "title": "Скрытый негатив — сарказм",
        "sentences": [
            "Ну и сколько мне ещё ждать?",
            "Ваш запрос в обработке",
            "Ну конечно, как всегда. Отличный сервис, ничего не скажешь.",
        ],
    },
    {
        "title": "Позитивный диалог — благодарность",
        "sentences": [
            "Получила заказ, всё идеально!",
            "Рады, что всё понравилось!",
            "Буду заказывать у вас ещё. Вы лучшие!",
        ],
    },
]

# Прогоняем через модель
print("=" * 70)
print("     ДЕМОНСТРАЦИЯ МОДЕЛИ РАСПОЗНАВАНИЯ НЕГАТИВА В ДИАЛОГАХ")
print("=" * 70)

for dialog in test_dialogs:
    print(f"\n{'─' * 70}")
    print(f"📋 {dialog['title']}")
    print(f"{'─' * 70}")

    for i, sentence in enumerate(dialog["sentences"], 1):
        print(f"  [{i}] {sentence}")

    result = predict_dialog(
        model, dialog["sentences"], tokenizer, device, threshold=train_config.threshold
    )

    print(f"\n  📊 Score негативности: {result['score']:.4f}")
    print(f"  🎯 Порог (threshold): {train_config.threshold}")
    print(f"  📌 Результат: {result['label_text']}")

    # Веса attention pooling
    weights = result["pooling_weights"]
    print(f"  📈 Веса внимания по репликам:")
    for i, w in enumerate(weights):
        bar = "█" * int(w * 40)
        print(f"      Реплика {i+1}: {w:.3f} |{bar}")

print(f"\n{'═' * 70}")
print("ДЕМОНСТРАЦИЯ ЗАВЕРШЕНА")
print(f"{'═' * 70}")

## 12. Визуализация Self-Attention между репликами

In [ ]:
def visualize_attention(model, sentences, tokenizer, device):
    """
    Визуализирует матрицу self-attention между репликами диалога.
    Показывает, как каждая реплика "смотрит" на остальные.
    """
    model.eval()

    # Подготовка
    all_input_ids = []
    all_attention_masks = []

    for sentence in sentences:
        encoded = tokenizer(
            sentence, padding="max_length", truncation=True,
            max_length=128, return_tensors="pt"
        )
        all_input_ids.append(encoded["input_ids"].squeeze(0))
        all_attention_masks.append(encoded["attention_mask"].squeeze(0))

    input_ids = torch.stack(all_input_ids).to(device)
    attention_mask = torch.stack(all_attention_masks).to(device)
    dialog_mask = torch.ones(1, len(sentences)).to(device)

    with torch.no_grad():
        scores, attn_weights, pooling_weights = model(
            input_ids, attention_mask, [len(sentences)], dialog_mask
        )

    # Attention weights: (batch, num_heads, seq_len, seq_len) или (batch, seq_len, seq_len)
    attn = attn_weights[0].cpu().numpy()
    if len(attn.shape) == 2:
        attn_matrix = attn[:len(sentences), :len(sentences)]
    else:
        # Усредняем по головам
        attn_matrix = attn.mean(axis=0)[:len(sentences), :len(sentences)]

    # Сокращаем реплики для отображения
    short_labels = [s[:35] + "..." if len(s) > 35 else s for s in sentences]

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        attn_matrix,
        xticklabels=short_labels,
        yticklabels=short_labels,
        cmap="YlOrRd",
        annot=True,
        fmt=".3f",
        annot_kws={"size": 10},
    )
    plt.title(f"Self-Attention между репликами\nScore: {scores.item():.4f}", fontsize=13)
    plt.xlabel("Key (на что смотрим)")
    plt.ylabel("Query (кто смотрит)")
    plt.xticks(rotation=25, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


# Пример негативного диалога
print("\n📊 Attention Map — негативный диалог:")
visualize_attention(
    model,
    [
        "Я оплатил заказ неделю назад",
        "Ваш заказ обрабатывается",
        "Неделю обрабатывается?! Это издевательство!",
        "Приносим извинения",
        "Мне не нужны извинения! Где мой товар?!",
    ],
    tokenizer,
    device,
)

# Пример нейтрального диалога
print("\n📊 Attention Map — нейтральный диалог:")
visualize_attention(
    model,
    [
        "Здравствуйте, подскажите стоимость",
        "Товар стоит 2500 рублей",
        "А доставка?",
        "Доставка бесплатная",
        "Отлично, оформляю! Спасибо!",
    ],
    tokenizer,
    device,
)

## 13. Распределение скоров на валидации

In [ ]:
# Собираем скоры для валидации
val_metrics_final, val_preds_final, val_labels_final, val_scores_final = evaluate(
    model, val_loader, device, train_config.threshold
)

# Визуализация распределения скоров
scores_neg = [s for s, l in zip(val_scores_final, val_labels_final) if l == 1]
scores_pos = [s for s, l in zip(val_scores_final, val_labels_final) if l == 0]

plt.figure(figsize=(10, 5))

if scores_pos:
    plt.hist(scores_pos, bins=15, alpha=0.7, label="Нейтральные", color="green")
if scores_neg:
    plt.hist(scores_neg, bins=15, alpha=0.7, label="Негативные", color="red")

plt.axvline(x=train_config.threshold, color="black", linestyle="--",
            linewidth=2, label=f"Threshold = {train_config.threshold}")
plt.xlabel("Score негативности", fontsize=12)
plt.ylabel("Количество диалогов", fontsize=12)
plt.title("Распределение скоров негативности на валидации", fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. Интерактивный режим

Введите свой диалог и получите предсказание модели.

In [ ]:
# Введите свой диалог здесь!
my_dialog = [
    "Здравствуйте, я получил не тот товар",
    "Извините, давайте разберёмся",
    "Мне прислали совсем другой размер! Это безответственность!",
    "Мы оформим возврат и отправим правильный",
    "Мне надоело ждать, я хочу возврат денег!",
]

print("═" * 60)
print("  ВАШИ ДАННЫЕ — ПРЕДСКАЗАНИЕ МОДЕЛИ")
print("═" * 60)
print("\nДиалог:")
for i, s in enumerate(my_dialog, 1):
    print(f"  [{i}] {s}")

result = predict_dialog(model, my_dialog, tokenizer, device, threshold=train_config.threshold)

print(f"\n{'─' * 60}")
print(f"  📊 Score: {result['score']:.4f}")
print(f"  🎯 Threshold: {train_config.threshold}")
print(f"  📌 Результат: {result['label_text']}")
print(f"{'─' * 60}")

# Веса
print("\n  Веса внимания (какие реплики важнее для решения):")
for i, w in enumerate(result["pooling_weights"]):
    bar = "█" * int(w * 50)
    print(f"    Реплика {i+1}: {w:.3f} |{bar}")

## 15. Сохранение модели

In [ ]:
# Сохраняем модель
os.makedirs("checkpoints", exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_config": model_config,
        "train_config": train_config,
        "best_f1": best_f1,
        "history": history,
    },
    "checkpoints/best_model.pt",
)

print("✓ Модель сохранена в checkpoints/best_model.pt")
print(f"  Лучший F1: {best_f1:.4f}")

## 16. Сводка по архитектуре (для защиты)

| # | Компонент | Описание |
|---|-----------|----------|
| 1 | **BERT Sentence Encoder** | Каждая реплика диалога кодируется через BERT (rubert-tiny2) в вектор размерности 312 с помощью mean pooling |
| 2 | **Positional Embedding** | К sentence embeddings добавляются позиционные эмбеддинги — порядок реплик в диалоге |
| 3 | **Multi-Head Self-Attention** | Каждая реплика «смотрит» на все остальные реплики диалога и обогащается контекстом (4 головы) |
| 4 | **Feed-Forward Network** | Нелинейное преобразование после attention (GELU + Dropout) |
| 5 | **Residual + LayerNorm** | Стабилизация обучения |
| 6 | **Attention Pooling** | Обучаемая агрегация контекстуализированных реплик в один вектор диалога |
| 7 | **Cosine Similarity (Dot Product)** | Нормализованное скалярное произведение вектора диалога с обучаемым вектором класса «негатив» |
| 8 | **Threshold** | Если score > 0.5 → диалог негативный |

### Ключевые преимущества:
- Учитывает контекст всего диалога, а не отдельных фраз
- Интерпретируемость через веса attention
- Работает с диалогами разной длины
- Использует предобученную языковую модель (transfer learning)